# Дообучение BERT: второй семинар
Алан-Барсаг Газзаев · 14 ноября 2026

Усиленный учебный опыт: полноразмерный RuBERT-base, больше обучающих данных и новая отложенная тестовая подвыборка. Результаты прежнего быстрого опыта с RuBERT-tiny2 сохранены отдельно.

Задача: классифицировать русские отзывы о кино как отрицательные, нейтральные или положительные. В первом семинаре мы применяли готовые головы. Теперь добавим новую голову к предобученному кодировщику и обучим модель на размеченных данных.

Основные инструменты: `load_dataset`, `Dataset.map`, `AutoModelForSequenceClassification`, `DataCollatorWithPadding`, `TrainingArguments`, `Trainer` и `pipeline`. Ручного цикла обучения нет.


## Запуск и маршрут
Рекомендуется Colab с **T4 GPU**. Выберите «Среда выполнения → Сменить среду выполнения → T4», затем выполните всё сверху вниз. Ключи API не нужны. Если установка обновила уже импортированные библиотеки, перезапустите сеанс и выполните всё заново.

План на 90 минут: постановка и данные — 20 минут; токенизация и базовые методы — 20; настройка Trainer и обучение — 25; оценка и ошибки — 15; сохранение и повторная загрузка — 10. Длительность скачивания зависит от сети.

Основной режим: до 9000 обучающих, 1000 валидационных и 600 тестовых отзывов, 3 эпохи. Это учебная подвыборка, а не полный benchmark. `SMOKE_RUN=True` сокращает данные и обучение для диагностики. Такой прогон нельзя выдавать за проверку основного режима.

Локально: создайте отдельное окружение, установите requirements.txt и запустите Jupyter. В этом выпуске проверяется запуск в Colab; локальное обучение отдельно не проверялось.


## 1. Окружение
Фиксируем совместимые версии, включая Accelerate, который используется Trainer. PyTorch в Colab уже установлен; для запуска нужен PyTorch не старее 2.6. Новые зависимости устанавливаются до импорта моделей.


In [ ]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
PACKAGES = {"transformers": "4.57.3", "datasets": "4.4.1", "accelerate": "1.12.0", "pandas": "2.2.3", "scikit-learn": "1.6.1", "matplotlib": "3.10.3"}
IMPORT_NAMES = {"scikit-learn": "sklearn"}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
needs = [f"{p}=={v}" for p,v in PACKAGES.items() if installed(p) != v]
loaded = any(IMPORT_NAMES.get(p,p) in sys.modules for p in PACKAGES if installed(p) != PACKAGES[p])
if needs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needs])
    if loaded:
        raise RuntimeError("Библиотеки обновлены. Перезапустите сеанс и выполните всё заново.")
print({p: installed(p) for p in PACKAGES})


In [ ]:
import os, json, time, gc, hashlib, platform, re
from pathlib import Path
from datetime import datetime, timezone
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, pipeline, set_seed)
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2,6)
pd.set_option("display.max_colwidth", 180)
SEED = 42
set_seed(SEED)
USE_GPU = torch.cuda.is_available()
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("Устройство:", torch.cuda.get_device_name(0) if USE_GPU else "CPU: обучение будет медленнее")


## 2. Условия эксперимента
Гиперпараметры задаём до просмотра теста. Кодировщик — `DeepPavlov/rubert-base-cased`, полноразмерная русскоязычная BERT-модель. При создании классификатора новая голова будет инициализирована случайно; её начальные ответы не являются качеством готового классификатора тональности.

Обновляем все параметры, без LoRA и без замораживания кодировщика. Лучший checkpoint выбирается по macro-F1 на validation. Ни выбор checkpoint, ни подбор параметров не используют test.


In [ ]:
SMOKE_RUN = False
LIMITS = {"train": 240, "validation": 90, "test": 90} if SMOKE_RUN else {"train": 9000, "validation": 1000, "test": 600}
EPOCHS = 1 if SMOKE_RUN else 3
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MODEL_ID = "DeepPavlov/rubert-base-cased"
MODEL_REVISION = "4036cab694767a299f2b9e6492909664d9414229"
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LABEL2ID = {v:k for k,v in ID2LABEL.items()}
RUN_DIR = Path("bert_finetuning_runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR.mkdir(parents=True)
START_TIME = time.time()
print("Режим:", "проверочный" if SMOKE_RUN else "основной", "Лимиты:", LIMITS, "Эпохи:", EPOCHS)
print("Каталог запуска:", RUN_DIR)


## 3. Данные и метки
Используем `ai-forever/kinopoisk-sentiment-classification`. В репозитории уже есть train, validation и test. Сохраняем их назначение: train обновляет веса, validation выбирает checkpoint, test используется для итоговой проверки.

Метки корпуса: 0 = Bad, 1 = Neutral, 2 = Good. В приложении называем их negative, neutral и positive. Не смешиваем эту схему со схемой другого корпуса из первого семинара.

Отзывы могут содержать субъективные оценки и разговорную лексику. Разметка не является объективной оценкой качества самого фильма.


In [ ]:
DATASET_ID = 'ai-forever/kinopoisk-sentiment-classification'
DATASET_REVISION = '4937df51b02a4c748b38bace5d749524fd90ae4a'
raw = load_dataset(DATASET_ID, revision=DATASET_REVISION)
assert {"train", "validation", "test"}.issubset(raw)
display(pd.DataFrame({"split": list(raw), "rows": [len(raw[k]) for k in raw]}))
display(raw["train"].select(range(3)).to_pandas()[["text", "label", "label_text"]])
expected_labels = {0: "Bad", 1: "Neutral", 2: "Good"}
assert all(expected_labels[int(x["label"])] == x["label_text"] for x in raw["train"])
print("Схема меток проверена по обучающим данным.")


## 4. Удаление точных повторов
Одинаковый текст в обучении и тесте создаёт утечку. Для проверки приводим пробелы и регистр к общему виду; исходный текст для модели не меняем. Сохраняем первое вхождение с приоритетом train, затем validation, затем test.

Пустые тексты и точные нормализованные повторы удаляются до выбора подвыборок. Это не поиск смысловых копий. Совпадение фильма, цитаты или пересказа без совпадения всей строки может остаться. Метки test не используются для изменения train.


In [ ]:
frames = []
for split in ["train", "validation", "test"]:
    frame = raw[split].to_pandas()
    frame["split"] = split
    frame["source_id"] = split + ":" + frame["id"].astype(str)
    frame["text"] = frame["text"].fillna("").astype(str)
    frame["text_key"] = frame["text"].str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()
    frames.append(frame)
all_rows = pd.concat(frames, ignore_index=True)
before_clean = all_rows.groupby("split").size()
all_rows = all_rows.loc[all_rows.text_key.str.len() > 0].drop_duplicates("text_key", keep="first")
clean = DatasetDict({s: Dataset.from_pandas(all_rows.loc[all_rows.split == s, ["source_id", "text", "label", "text_key"]], preserve_index=False) for s in ["train", "validation", "test"]})
cleaning_summary = pd.DataFrame({"before": before_clean, "after": {s: len(clean[s]) for s in clean}})
display(cleaning_summary)
key_sets = {s: set(clean[s]["text_key"]) for s in clean}
assert not (key_sets["train"] & key_sets["validation"])
assert not (key_sets["train"] & key_sets["test"])
assert not (key_sets["validation"] & key_sets["test"])
print("Точные нормализованные пересечения между частями отсутствуют.")


### Учебные подвыборки и воспроизводимость
Перемешиваем каждую официальную часть с одним seed и берём ограниченное число записей. Первые 600 перемешанных тестовых отзывов уже использовались в коротком опыте с tiny-моделью и исключены. Новая тестовая подвыборка — следующие 600 записей; её состав фиксируется до обучения. Это случайная учебная подвыборка, а не полный датасет. Проверяем наличие всех классов.

Для воспроизведения сохраняются source_id и хеши текстов. Test не читаем для поиска удачных примеров и не используем для выбора настроек.


In [ ]:
# Первые 600 записей test уже оценивались в первом опыте: исключаем их.
shuffled = DatasetDict({s: clean[s].shuffle(seed=SEED) for s in clean})
previous_test_ids = set(shuffled["test"].select(range(600))["source_id"])
assert len(shuffled["test"]) >= 1200
splits = DatasetDict({
    "train": shuffled["train"].select(range(min(LIMITS["train"], len(shuffled["train"])))),
    "validation": shuffled["validation"].select(range(min(LIMITS["validation"], len(shuffled["validation"])))),
    "test": shuffled["test"].select(range(600, 600 + LIMITS["test"])),
})
assert not (previous_test_ids & set(splits["test"]["source_id"]))
class_counts = pd.DataFrame({s: pd.Series(splits[s]["label"]).value_counts().reindex([0,1,2], fill_value=0) for s in splits})
class_counts.index = [ID2LABEL[i] for i in class_counts.index]
display(class_counts)
assert (class_counts > 0).all().all()
split_manifest = {s: [{"source_id": row["source_id"], "text_sha256": hashlib.sha256(row["text_key"].encode()).hexdigest()} for row in splits[s]] for s in splits}
(RUN_DIR/"split_manifest.json").write_text(json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print({s: len(splits[s]) for s in splits})


## 5. Токенизация через Dataset.map
Токенизатор загружается из того же checkpoint, что и модель. `map(..., batched=True)` преобразует сразу группу текстов. На этом этапе не добавляем PAD: `DataCollatorWithPadding` сделает это отдельно внутри каждого батча.

Предел 256 токенов выбран для скорости семинара. Длинные отзывы обрезаются: итог относится к этой постановке, а не к пониманию всего отзыва. Сохраним видимую часть после декодирования, чтобы TF-IDF сравнивался на том же ограниченном содержимом, а не получал полный текст.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
def tokenize_batch(batch):
    encoded = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    encoded["visible_text"] = tokenizer.batch_decode(encoded["input_ids"], skip_special_tokens=True)
    return encoded
prepared = splits.map(tokenize_batch, batched=True, desc="Токенизация")
token_columns = ["input_ids", "attention_mask", "token_type_ids", "label"]
tokenized = DatasetDict({s: prepared[s].select_columns([c for c in token_columns if c in prepared[s].column_names]) for s in prepared})
full_lengths = [len(x) for x in tokenizer(list(splits["validation"]["text"]), truncation=False, add_special_tokens=True, verbose=False)["input_ids"]]
truncation_rate = float(np.mean(np.array(full_lengths) > MAX_LENGTH))
print("Доля обрезанных текстов на validation:", round(truncation_rate, 3))
display(pd.DataFrame({"original_start": [prepared["train"][0]["text"][:400]], "visible_start": [prepared["train"][0]["visible_text"][:400]]}))
assert max(len(x) for x in tokenized["train"]["input_ids"]) <= MAX_LENGTH


### Динамическое заполнение батча
Коллатор собирает переменные длины в прямоугольный тензор, создаёт labels и добавляет заполнение. Для коротких примеров это уменьшает бесполезные вычисления по сравнению с заполнением каждого текста до общего максимума.

Проверяем формы входов и целей, но не пишем собственный DataLoader и не реализуем перенос данных вручную.


In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if USE_GPU else None)
example_batch = collator([tokenized["train"][i] for i in range(4)])
print({k: tuple(v.shape) for k,v in example_batch.items()})
assert example_batch["input_ids"].shape == example_batch["attention_mask"].shape
assert example_batch["labels"].shape == (4,)
del example_batch


## 6. Базовые методы
Сначала оцениваем константный классификатор и TF-IDF с логистической регрессией. Это помогает понять, насколько полезно сложное обучение относительно простого метода.

TF-IDF обучает словарь только на train. Коэффициент C выбирается из 0.5, 1 и 2 по validation macro-F1. Максимум признаков увеличен до 50000 для сильного базового сравнения. Validation лишь преобразуется через уже обученный pipeline. Используются те же видимые префиксы, что и у BERT. Сравнение всё равно относится к целым методам: представления и процедуры обучения различаются.


In [ ]:
def scores(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)), "macro_f1": float(f1_score(y_true, y_pred, labels=[0,1,2], average="macro", zero_division=0))}
x_train = list(prepared["train"]["visible_text"])
x_val = list(prepared["validation"]["visible_text"])
y_train = np.array(prepared["train"]["label"])
y_val = np.array(prepared["validation"]["label"])
dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train),1)), y_train)
dummy_val = dummy.predict(np.zeros((len(y_val),1)))
# Настраиваем базовый метод по validation, не ослабляя его ради BERT.
tfidf_candidates = []
tfidf_search_rows = []
for C in [0.5, 1.0, 2.0]:
    candidate = make_pipeline(TfidfVectorizer(ngram_range=(1,2), max_features=50000, min_df=2), LogisticRegression(C=C, max_iter=1000, random_state=SEED))
    candidate.fit(x_train, y_train)
    candidate_pred = candidate.predict(x_val)
    tfidf_search_rows.append({"C": C, **scores(y_val, candidate_pred)})
    tfidf_candidates.append(candidate)
best_tfidf_index = int(np.argmax([r["macro_f1"] for r in tfidf_search_rows]))
tfidf = tfidf_candidates[best_tfidf_index]
tfidf_val = tfidf.predict(x_val)
display(pd.DataFrame(tfidf_search_rows))
validation_rows = [{"method": "Константный класс", **scores(y_val, dummy_val)}, {"method": "TF-IDF + LogisticRegression", **scores(y_val, tfidf_val)}]
display(pd.DataFrame(validation_rows))


## 7. Новая классификационная голова
Загружаем предобученный кодировщик и создаём голову из трёх классов. Сообщение о новых весах classifier ожидаемо: этих весов не было в checkpoint базовой модели. До обучения её ответы не следует использовать как готовый классификатор.

Все параметры остаются обучаемыми. Инициализация и порядок данных воспроизводятся через seed, но полное побитовое совпадение на разных GPU и версиях библиотек не обещается.


In [ ]:
set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, revision=MODEL_REVISION, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
print("Всего параметров:", sum(p.numel() for p in model.parameters()))
print("Обучаемых:", sum(p.numel() for p in model.parameters() if p.requires_grad))
head_before = model.classifier.weight.detach().cpu().clone()
encoder_before = model.bert.encoder.layer[0].attention.self.query.weight.detach().cpu().clone()
assert all(p.requires_grad for p in model.parameters())


## 8. TrainingArguments и Trainer
Эпоха — один проход по train. После каждой эпохи Trainer оценивает validation и сохраняет checkpoint. Лучший вариант выбирается по macro-F1, а не по минимальной ошибке на test.

Параметры: learning rate 2e-5, weight decay 0.01, batch size 16. FP16 включается только на CUDA. Логи остаются в ноутбуке: внешние системы отслеживания и отправка модели на Hub отключены.

Функция compute_metrics только переводит логиты в классы и считает метрики. Обратное распространение, оптимизатор и цикл обучения выполняет Trainer.


In [ ]:
def compute_metrics(prediction):
    logits, labels = prediction
    return scores(labels, np.argmax(logits, axis=-1))
args = TrainingArguments(
    output_dir=str(RUN_DIR/"checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    fp16=USE_GPU,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
    push_to_hub=False,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"], processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
expected_steps = int(np.ceil(len(tokenized["train"])/BATCH_SIZE)*EPOCHS)
print("Ожидаемые шаги оптимизатора:", expected_steps)
print("Выбор лучшей модели: validation macro-F1.")


### Оценка до обучения
Получим предсказания модели с новой случайной головой на validation. Сравнение «до/после» показывает эффект адаптации к классам задачи. Оно не сравнивает два уже готовых продукта и не доказывает преимущество BERT над всеми альтернативами.


In [ ]:
before_output = trainer.predict(tokenized["validation"], metric_key_prefix="before")
before_pred = before_output.predictions.argmax(axis=-1)
before_scores = scores(y_val, before_pred)
validation_rows.append({"method": "BERT до обучения новой головы", **before_scores})
display(pd.DataFrame(validation_rows))


## 9. Обучение одним вызовом
Во время обучения смотрите на loss и validation macro-F1. Уменьшение train loss без улучшения validation может означать переобучение. Лучший checkpoint может оказаться не последним.

Не перезапускайте эту ячейку для дополнительного обучения на тех же весах, если хотите воспроизвести исходный опыт: для нового независимого запуска нужно начать с чистого ядра. При прерывании текущий прогон считается незавершённым.


In [ ]:
train_started = time.time()
train_output = trainer.train()
training_seconds = time.time()-train_started
assert trainer.state.global_step == expected_steps
assert trainer.state.best_model_checkpoint is not None
assert np.isfinite(train_output.training_loss)
assert not torch.equal(head_before, trainer.model.classifier.weight.detach().cpu())
assert not torch.equal(encoder_before, trainer.model.bert.encoder.layer[0].attention.self.query.weight.detach().cpu())
print("Обучение завершено. Шагов:", trainer.state.global_step)
print("Время обучения, секунд:", round(training_seconds,1))
print("Лучший checkpoint:", trainer.state.best_model_checkpoint)
print("Лучшая validation macro-F1:", trainer.state.best_metric)


## 10. История и результат на validation
Сопоставляем модели на одной выборке, с теми же метками. Улучшение относительно случайной головы ожидаемо, но превосходство над TF-IDF не гарантируется. Небольшой объём обучения и обрезка длинных отзывов ограничивают результат.

График показывает оценки после эпох. Он не является кривой по test. Случайный seed выбран один, поэтому различия не интерпретируем как статистически доказанное превосходство.


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
validation_history = history.loc[history["eval_macro_f1"].notna(), ["epoch", "step", "eval_loss", "eval_accuracy", "eval_macro_f1"]]
display(validation_history)
validation_history.plot(x="epoch", y="eval_macro_f1", marker="o", ylim=(0,1), title="Validation macro-F1 после эпох")
plt.ylabel("Macro-F1")
plt.show()
after_output = trainer.predict(tokenized["validation"], metric_key_prefix="after")
after_pred = after_output.predictions.argmax(axis=-1)
after_scores = scores(y_val, after_pred)
validation_rows.append({"method": "BERT после дообучения", **after_scores})
validation_comparison = pd.DataFrame(validation_rows)
display(validation_comparison)
print("Изменение macro-F1 относительно новой головы:", round(after_scores["macro_f1"]-before_scores["macro_f1"],4))


### Разбор ошибок без подгонки теста
Показываем фрагмент текста, истинную и предсказанную метку. Если общий отзыв длинный, учитываем, что модель видела только префикс. Уверенный неправильный ответ всё равно является ошибкой.

Разбор validation допустим при разработке. Но после изменения условий нужно повторить эксперимент целиком и честно отметить, что выбран новый вариант.


In [ ]:
val_probabilities = torch.softmax(torch.tensor(after_output.predictions), dim=-1).numpy()
val_errors = pd.DataFrame({"source_id": prepared["validation"]["source_id"], "visible_text": x_val, "gold": [ID2LABEL[int(x)] for x in y_val], "prediction": [ID2LABEL[int(x)] for x in after_pred], "score": val_probabilities.max(axis=-1)})
val_errors = val_errors.loc[y_val != after_pred].sort_values("score", ascending=False)
display(val_errors.head(6))
print("Ошибок на validation:", len(val_errors), "из", len(y_val))
display(pd.DataFrame(confusion_matrix(y_val, after_pred, labels=[0,1,2]), index=ID2LABEL.values(), columns=ID2LABEL.values()))


## 11. Фиксация решения и итоговый test
До этой ячейки test не использовался для оценки качества и выбора checkpoint. Теперь фиксируем настройки и оцениваем дообученный BERT и базовые методы на одной отложенной части.

После просмотра test нельзя продолжать подбор параметров по этому же набору и называть его результат независимым. Корпус предобучения RuBERT не аудирован на пересечения с отзывами, поэтому абсолютная независимость от предобучения не установлена.


In [ ]:
frozen_choice = {"model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dataset_id": DATASET_ID, "dataset_revision": DATASET_REVISION, "max_length": MAX_LENGTH, "seed": SEED, "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "tfidf_C": tfidf_search_rows[best_tfidf_index]["C"], "test_offset": 600, "best_checkpoint": trainer.state.best_model_checkpoint, "selected_using": "validation macro_f1", "smoke_run": SMOKE_RUN}
(RUN_DIR/"frozen_choice.json").write_text(json.dumps(frozen_choice,ensure_ascii=False,indent=2),encoding="utf-8")
test_output = trainer.predict(tokenized["test"], metric_key_prefix="test")
test_pred = test_output.predictions.argmax(axis=-1)
y_test = np.array(prepared["test"]["label"])
tfidf_test = tfidf.predict(list(prepared["test"]["visible_text"]))
dummy_test = dummy.predict(np.zeros((len(y_test),1)))
test_comparison = pd.DataFrame([{"method": "Константный класс", **scores(y_test,dummy_test)}, {"method": "TF-IDF + LogisticRegression", **scores(y_test,tfidf_test)}, {"method": "Дообученный BERT", **scores(y_test,test_pred)}])
display(test_comparison)
print(classification_report(y_test,test_pred,labels=[0,1,2],target_names=list(ID2LABEL.values()),zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test,test_pred,labels=[0,1,2]),index=ID2LABEL.values(),columns=ID2LABEL.values()))
assert len(test_pred)==len(prepared["test"])
assert np.isfinite(test_output.predictions).all()
assert set(test_pred).issubset({0,1,2})


## 12. Сохранение модели и повторная загрузка
Сохраняем выбранные веса и токенизатор вместе. После загрузки через обычный `pipeline()` модель должна работать без объекта Trainer. Это проверка того, что мы получили пригодный для применения артефакт.

Для проверки совпадения используем несколько validation-примеров. Повторная загрузка не является дополнительным подбором по test. Небольшие численные различия score возможны из-за точности вычислений; сравниваем выбранные классы.


In [ ]:
MODEL_DIR = RUN_DIR/"model"
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
assert (MODEL_DIR/"config.json").exists()
assert (MODEL_DIR/"model.safetensors").exists()
# RuBERT использует WordPiece, а не токенизатор Mistral.
# В Transformers 4.57.3 локальный словарь >100000 вызывает ложное предупреждение.
saved_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), fix_mistral_regex=False)
probe_texts = list(prepared["validation"].select(range(8))["text"])
assert tokenizer(probe_texts, truncation=True, max_length=MAX_LENGTH)["input_ids"] == saved_tokenizer(probe_texts, truncation=True, max_length=MAX_LENGTH)["input_ids"]
if "saved_classifier" in globals():
    del saved_classifier
    gc.collect()
    if USE_GPU: torch.cuda.empty_cache()
saved_classifier = pipeline("text-classification", model=str(MODEL_DIR), tokenizer=saved_tokenizer, device=0 if USE_GPU else -1)
reloaded = saved_classifier(probe_texts, truncation=True, max_length=MAX_LENGTH, batch_size=8)
reloaded_labels = [LABEL2ID[x["label"]] for x in reloaded]
assert reloaded_labels == after_pred[:8].tolist(), "Классы после загрузки отличаются: требуется проверка артефакта"
print("Повторная загрузка: классы совпали на 8 validation-примерах.")
fresh_texts = ["Великолепный фильм, прекрасная актёрская игра.", "Скучное кино, совершенно не понравилось.", "Фильм вышел на экраны в прошлом году."]
fresh_results = saved_classifier(fresh_texts, truncation=True,max_length=MAX_LENGTH,batch_size=3)
display(pd.DataFrame([{"text":text,**result} for text,result in zip(fresh_texts,fresh_results)]))


## 13. Отчёт и архивы
Отдельно сохраняем компактный отчёт с метриками и архив модели с токенизатором. В отчёт входят условия эксперимента, размеры частей, ревизии, история обучения, предсказания и ошибки validation.

Архив модели пригоден для повторной загрузки. Checkpoint с состоянием оптимизатора нужен для продолжения обучения и остаётся отдельно в каталоге checkpoints. Он не включается в лёгкий архив отчёта. Публикация на Hub автоматически не выполняется.


In [ ]:
REPORT_DIR = RUN_DIR/"report"
REPORT_DIR.mkdir(exist_ok=True)
validation_comparison.to_csv(REPORT_DIR/"validation_metrics.csv",index=False)
test_comparison.to_csv(REPORT_DIR/"test_metrics.csv",index=False)
validation_history.to_csv(REPORT_DIR/"training_history.csv",index=False)
val_errors.to_csv(REPORT_DIR/"validation_errors.csv",index=False)
pd.DataFrame({"source_id":prepared["test"]["source_id"],"gold":y_test,"bert":test_pred,"tfidf":tfidf_test}).to_csv(REPORT_DIR/"test_predictions.csv",index=False)
report = {"author":"Алан-Барсаг Газзаев","experiment":frozen_choice,"sizes":{s:len(splits[s]) for s in splits},"environment":{"python":platform.python_version(),"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0) if USE_GPU else None,"packages":{p:installed(p) for p in PACKAGES}},"training_seconds":training_seconds,"total_seconds":time.time()-START_TIME,"optimizer_steps":trainer.state.global_step,"validation":validation_rows,"test":test_comparison.to_dict("records"),"validation_truncation_rate":truncation_rate,"checks":{"exact_text_overlap":False,"all_parameters_trainable":True,"encoder_and_head_updated":True,"completed_expected_steps":True,"reload_predictions_match":True,"reload_token_ids_match":True},"limitations":["Один seed","Учебные подвыборки","Обрезка до 256 токенов","Смысловые дубликаты не исключены","Пересечение с предобучением не аудировано"]}
(REPORT_DIR/"report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
import shutil
shutil.copy2(RUN_DIR/"split_manifest.json",REPORT_DIR/"split_manifest.json")
report_archive = shutil.make_archive(str(RUN_DIR/"training_report"),"zip",REPORT_DIR)
model_archive = shutil.make_archive(str(RUN_DIR/"trained_model"),"zip",MODEL_DIR)
print("ГОТОВО: обучение, итоговый test и повторная загрузка выполнены.")
print("Отчёт:", REPORT_DIR/"report.json")
print("Архив отчёта:", report_archive)
print("Архив модели:", model_archive)


## Самостоятельная работа
1. На новом запуске измените одну настройку: learning rate, число эпох или предел длины. Заранее сформулируйте гипотезу.
2. Сравнивайте варианты на validation. Для окончательной оценки используйте новый, заранее отложенный набор, если текущий test уже участвовал в принятии решений.
3. Найдите две ошибки, связанные с неоднозначной оценкой, и две возможные ошибки из-за обрезки. Не делайте причинный вывод только по одному примеру.
4. Сравните обучение только головы с полным дообучением как отдельный опыт. Основной notebook обучает все веса.
5. Объясните, почему хороший результат после обучения новой головы не доказывает превосходство над TF-IDF или большой готовой моделью.

## Диагностика
- Ошибка импорта после установки: перезапустить сеанс и выполнить все ячейки.
- Не хватает памяти: уменьшить batch size и начать новый прогон; зафиксировать изменение.
- Нет GPU: включить T4; CPU возможен, но значительно медленнее.
- Предупреждение о новых весах classifier при первом создании: ожидаемо, голову мы обучаем.
- Потери NaN или шаги не завершены: прогон не прошёл проверку, не используйте его как готовый результат.
- Отключилась среда: сохраните архивы заранее; файлы /content исчезают при удалении среды.

[Transformers: классификация](https://huggingface.co/docs/transformers/v4.57.3/en/tasks/sequence_classification) · [Trainer](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/trainer) · [Датасет](https://huggingface.co/datasets/ai-forever/kinopoisk-sentiment-classification) · [RuBERT-base](https://huggingface.co/DeepPavlov/rubert-base-cased)
